# Working with a GTFS feed

Once you have a GTFS feed, transitio can validate it, report what it found,
repair it and edit it. This page uses the feed of Turku's public transport,
which the [Quickstart](quickstart.ipynb) fetched. Let's fetch it again;
`fetch` delivers the copy in its download cache:

In [1]:
import transitio

rec = transitio.place("Turku").recommend("2026-10-14")
result = transitio.fetch(feeds=rec, osm=False, progress=False)
path = result.feeds[0]
path.name

'c267955299d3f74ba34cfbef1d962f4421a1757af444c8e31ab935bbbb448f30-cropped.zip'

## Validate a feed

`validate_feed` checks a feed with transitio's Rust core, which emits the
notice codes of the canonical
[gtfs-validator](https://gtfs-validator.mobilitydata.org/). It returns a
dictionary with the notices, the row count of each file and the service
window, among others:

In [2]:
validation = transitio.validate_feed(path)
validation["row_counts"]

{'agency.txt': 4,
 'calendar.txt': 1034,
 'calendar_dates.txt': 66212,
 'feed_info.txt': 1,
 'routes.txt': 99,
 'shapes.txt': 249064,
 'stop_times.txt': 699887,
 'stops.txt': 2954,
 'translations.txt': 1941,
 'trips.txt': 15687}

## Read the validation report

`transitio.report.build_report` turns a validation into a report, merged with
the hosted canonical-validator report when you pass one (`hosted=`):

In [3]:
report = transitio.report.build_report(validation)
for group in report["notices"]:
    print(group["code"], group["severity"], group["source"], group["totalNotices"])

equal_shape_distance_diff_coordinates ERROR local 1751
equal_shape_distance_same_coordinates WARNING local 10000
service_has_no_active_day_of_the_week WARNING local 1034
expired_calendar WARNING local 44
notice_limit_reached WARNING local 1
no_fare_information INFO local 1
unknown_file INFO local 1


Notices follow the canonical
[gtfs-validator](https://gtfs-validator.mobilitydata.org/) codes and
severities, so local and hosted results merge into one document. Each notice
group records whether it was seen locally, by the hosted validator, or both.

`report["summary"]` carries the severity totals (`counts`), the computed
service window (`serviceWindow`, the actual calendar activity, not the
published range), row counts (`rowCounts`) and the provenance block of the
downloaded dataset (`provenance`):

In [4]:
summary = report["summary"]
print(summary["counts"])
print(summary["serviceWindow"])

{'errors': 1751, 'warnings': 11079, 'infos': 2}
['20260810', '20270606']


`transitio.report.render_markdown(report)` and `render_html(report)` write
the report out as a document.

## Repair a feed

`repair_feed` applies the fixes that keep the feed equivalent from the
passenger's point of view, under the [gtfstidy](https://github.com/patrickbr/gtfstidy)
contract, and logs every fix. It stops at 10,000 notices in one file unless
told otherwise, and Turku's shapes.txt has more, so we raise the limit:

In [5]:
repair = transitio.repair_feed(path, "repaired.zip", max_notices_per_file=100_000)
print(len(repair["fixes"]), "fixes;", len(repair["remaining_notices"]), "notices remain")

0 fixes; 15191 notices remain


For Turku's feed the repair finds nothing to fix: none of its notices is one
the repair handles. `repair["fixes"]` lists each fix with its action, its
location and the notice that triggered it, and `remaining_notices` comes from
validating the repaired feed again.

## Edit a feed in the GUI

```
pip install "transitio-editor[snap]"
transitio edit feed.zip --osm-pbf helsinki.osm.pbf
```

The editor lives in its own package,
[transitio-editor](https://github.com/cafein-py/transitio-editor); the
`transitio edit` command delegates to it.

The editor opens on `http://127.0.0.1:8300`: stops and shapes render on
a map where they can be added, moved and drawn — with the `--osm-pbf`
extract, drawn route shapes snap to the street network, Remix-style.
Saving runs the validator and reports the notice counts. The HTTP API
behind the interface publishes its schema at `/openapi.json`.